# SCM Item Master Tool-calling Agent

This is an AI Playground-exported notebook adapted for the SCM Item Master use case. In this notebook, you will:
- Author a tool-calling [MLflow's `ResponsesAgent`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.pyfunc.html#mlflow.pyfunc.ResponsesAgent) that uses the OpenAI client
- Manually test the agent's output
- Evaluate the agent with Mosaic AI Agent Evaluation
- Log and deploy the agent

This notebook should be run on serverless or a cluster with DBR<17.

 **_NOTE:_**  This notebook uses the OpenAI SDK, but AI Agent Framework is compatible with any agent authoring framework, including LlamaIndex or LangGraph. To learn more, see the [Authoring Agents](https://docs.databricks.com/generative-ai/agent-framework/author-agent) Databricks documentation.

## Prerequisites

- Address all `TODO`s in this notebook.

In [0]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents mlflow-skinny[databricks]
dbutils.library.restartPython()

## Define the agent in code
Below we define our agent code in a single cell, enabling us to easily write it to a local Python file for subsequent logging and deployment using the `%%writefile` magic command.

For more examples of tools to add to your agent, see [docs](https://docs.databricks.com/generative-ai/agent-framework/agent-tool.html).

In [0]:
%%writefile agent.py
import json
from typing import Any, Callable, Generator, Optional
from uuid import uuid4
import warnings

import backoff
import mlflow
import openai
from databricks.sdk import WorkspaceClient
from databricks_openai import UCFunctionToolkit, VectorSearchRetrieverTool
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from openai import OpenAI
from pydantic import BaseModel
from unitycatalog.ai.core.base import get_uc_function_client

############################################
# Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"

SYSTEM_PROMPT = """You are a supply-chain assistant for a fictional laptop and server manufacturer. Use search_item_master to find relevant item records by meaning. Use get_inventory for exact quantities at a named warehouse. Use get_supplier for supplier name, supplier city, supplier country, and lead time. Do not invent missing facts. If a required item ID or warehouse is missing, ask a clarifying question. If the user asks for total stock across all locations, do not assume a warehouse; use an appropriate all-locations query/tool if one is available. Do not call a single-warehouse inventory tool as though it returns a total. Supplier location and warehouse location are different. Similarity does not prove technical compatibility; do not claim an item is a compatible substitute unless the data explicitly establishes that."""


###############################################################################
## Define tools for your agent, enabling it to retrieve data or take actions
## beyond text generation
## To create and see usage examples of more tools, see
## https://docs.databricks.com/generative-ai/agent-framework/agent-tool.html
###############################################################################
class ToolInfo(BaseModel):
    """
    Class representing a tool for the agent.
    - "name" (str): The name of the tool.
    - "spec" (dict): JSON description of the tool (matches OpenAI Responses format)
    - "exec_fn" (Callable): Function that implements the tool logic
    """

    name: str
    spec: dict
    exec_fn: Callable


def create_tool_info(tool_spec, exec_fn_param: Optional[Callable] = None):
    tool_spec["function"].pop("strict", None)
    tool_name = tool_spec["function"]["name"]
    udf_name = tool_name.replace("__", ".")

    # Define a wrapper that accepts kwargs for the UC tool call,
    # then passes them to the UC tool execution client
    def exec_fn(**kwargs):
        function_result = uc_function_client.execute_function(udf_name, kwargs)
        if function_result.error is not None:
            return function_result.error
        else:
            return function_result.value
    return ToolInfo(name=tool_name, spec=tool_spec, exec_fn=exec_fn_param or exec_fn)


TOOL_INFOS = []

# You can use UDFs in Unity Catalog as agent tools
# TODO: Add additional tools
UC_TOOL_NAMES = ["main.scm_e2e_rag_lab.search_item_master", "main.scm_e2e_rag_lab.get_inventory", "main.scm_e2e_rag_lab.get_supplier"]

uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
uc_function_client = get_uc_function_client()
for tool_spec in uc_toolkit.tools:
    TOOL_INFOS.append(create_tool_info(tool_spec))


# Use Databricks vector search indexes as tools
# See [docs](https://docs.databricks.com/generative-ai/agent-framework/unstructured-retrieval-tools.html) for details

# # (Optional) Use Databricks vector search indexes as tools
# # See https://docs.databricks.com/generative-ai/agent-framework/unstructured-retrieval-tools.html
# # for details
VECTOR_SEARCH_TOOLS = []
# # TODO: Add vector search indexes as tools or delete this block
# VECTOR_SEARCH_TOOLS.append(
#         VectorSearchRetrieverTool(
#         index_name="",
#         # filters="..."
#     )
# )
for vs_tool in VECTOR_SEARCH_TOOLS:
    TOOL_INFOS.append(create_tool_info(vs_tool.tool, vs_tool.execute))



class ToolCallingAgent(ResponsesAgent):
    """
    Class representing a tool-calling Agent
    """

    def __init__(self, llm_endpoint: str, tools: list[ToolInfo]):
        """Initializes the ToolCallingAgent with tools."""
        self.llm_endpoint = llm_endpoint
        self.workspace_client = WorkspaceClient()
        self.model_serving_client: OpenAI = (
            self.workspace_client.serving_endpoints.get_open_ai_client()
        )
        self._tools_dict = {tool.name: tool for tool in tools}

    def get_tool_specs(self) -> list[dict]:
        """Returns tool specifications in the format OpenAI expects."""
        return [tool_info.spec for tool_info in self._tools_dict.values()]

    @mlflow.trace(span_type=SpanType.TOOL)
    def execute_tool(self, tool_name: str, args: dict) -> Any:
        """Executes the specified tool with the given arguments."""
        return self._tools_dict[tool_name].exec_fn(**args)

    @staticmethod
    def _merge_consecutive_assistant_messages(cc_messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
        # The canonical OpenAI shape for parallel tool calls is one assistant
        # message whose tool_calls array lists every call in that turn.
        # to_chat_completions_input emits a separate assistant message per
        # function_call, which some providers reject. Collapse consecutive
        # assistant messages so all tool_calls from one turn share a single
        # message.
        merged: list[dict[str, Any]] = []
        for msg in cc_messages:
            if (
                msg.get("role") == "assistant"
                and merged
                and merged[-1].get("role") == "assistant"
            ):
                prev = merged[-1]
                prev_text = prev.get("content") if prev.get("content") not in (None, "", "tool call") else None
                cur_text = msg.get("content") if msg.get("content") not in (None, "", "tool call") else None
                if prev_text and cur_text:
                    prev["content"] = prev_text + "\n" + cur_text
                elif cur_text:
                    prev["content"] = cur_text
                if msg.get("tool_calls"):
                    prev["tool_calls"] = (prev.get("tool_calls") or []) + msg["tool_calls"]
            else:
                merged.append(dict(msg))
        return merged

    def call_llm(self, messages: list[dict[str, Any]]) -> Generator[dict[str, Any], None, None]:
        cc_messages = self._merge_consecutive_assistant_messages(to_chat_completions_input(messages))
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="PydanticSerializationUnexpectedValue")
            for chunk in self.model_serving_client.chat.completions.create(
                model=self.llm_endpoint,
                messages=cc_messages,
                tools=self.get_tool_specs(),
                stream=True,
            ):
                chunk_dict = chunk.to_dict()
                if len(chunk_dict.get("choices", [])) > 0:
                    yield chunk_dict

    def handle_tool_call(
        self,
        tool_call: dict[str, Any],
        messages: list[dict[str, Any]],
    ) -> ResponsesAgentStreamEvent:
        """
        Execute tool calls, add them to the running message history, and return a ResponsesStreamEvent w/ tool output
        """
        try:
            args = json.loads(tool_call.get("arguments"))
        except Exception as e:
            args = {}
        result = str(self.execute_tool(tool_name=tool_call["name"], args=args))

        tool_call_output = self.create_function_call_output_item(tool_call["call_id"], result)
        messages.append(tool_call_output)
        return ResponsesAgentStreamEvent(type="response.output_item.done", item=tool_call_output)

    def call_and_run_tools(
        self,
        messages: list[dict[str, Any]],
        max_iter: int = 10,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        for _ in range(max_iter):
            # The LLM may emit multiple tool calls in a single turn (parallel
            # tool calls). The next LLM request must include a tool_result for
            # every tool_use emitted in the previous turn; missing any one
            # causes the request to be rejected. Before going back to the LLM,
            # execute every function_call whose call_id has no matching
            # function_call_output yet.
            handled = {m["call_id"] for m in messages if m.get("type") == "function_call_output"}
            pending = [m for m in messages if m.get("type") == "function_call" and m["call_id"] not in handled]
            if pending:
                for call in pending:
                    yield self.handle_tool_call(call, messages)
                continue

            last_msg = messages[-1]
            if last_msg.get("type") == "message" and last_msg.get("role") == "assistant":
                return

            yield from output_to_responses_items_stream(
                chunks=self.call_llm(messages), aggregator=messages
            )

        yield ResponsesAgentStreamEvent(
            type="response.output_item.done",
            item=self.create_text_output_item("Max iterations reached. Stopping.", str(uuid4())),
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id

        if session_id:
            mlflow.update_current_trace(
                metadata={
                    "mlflow.trace.session": session_id,
                }
            )

        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(self, request: ResponsesAgentRequest) -> Generator[ResponsesAgentStreamEvent, None, None]:
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id

        if session_id:
            mlflow.update_current_trace(
                metadata={
                    "mlflow.trace.session": session_id,
                }
            )

        messages = to_chat_completions_input([i.model_dump() for i in request.input])
        if SYSTEM_PROMPT:
            messages.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
        yield from self.call_and_run_tools(messages=messages)


# Log the model using MLflow
mlflow.openai.autolog()
AGENT = ToolCallingAgent(llm_endpoint=LLM_ENDPOINT_NAME, tools=TOOL_INFOS)
mlflow.models.set_model(AGENT)

## Test the agent

Interact with the agent to test its output. Since we manually traced methods within `ResponsesAgent`, you can view the trace for each step the agent takes, with any LLM calls made via the OpenAI SDK automatically traced by autologging.

Replace this placeholder input with an appropriate domain-specific example for your agent.

In [0]:
dbutils.library.restartPython()

In [0]:
from agent import AGENT

AGENT.predict(
    {"input": [{"role": "user", "content": "Find the 1 TB SSD, check its stock in Chennai, and tell me the supplier’s city, country, and lead time."}], "custom_inputs": {"session_id": "test-session-123"}},
)

In [0]:
for chunk in AGENT.predict_stream(
    {"input": [{"role": "user", "content": "Find the 1 TB SSD, check its stock in Chennai, and tell me the supplier’s city, country, and lead time."}], "custom_inputs": {"session_id": "test-session-123"}}
):
    print(chunk.model_dump(exclude_none=True))

### Log the `agent` as an MLflow model
Determine Databricks resources to specify for automatic auth passthrough at deployment time
- **TODO**: If your Unity Catalog Function queries a [vector search index](https://docs.databricks.com/generative-ai/agent-framework/unstructured-retrieval-tools.html) or leverages [external functions](https://docs.databricks.com/generative-ai/agent-framework/external-connection-tools.html), you need to include the dependent vector search index and UC connection objects, respectively, as resources. See [docs](https://docs.databricks.com/generative-ai/agent-framework/log-agent.html#specify-resources-for-automatic-authentication-passthrough) for more details.

Log the agent as code from the `agent.py` file. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
import mlflow
from agent import LLM_ENDPOINT_NAME, VECTOR_SEARCH_TOOLS, uc_toolkit
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from pkg_resources import get_distribution

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)]
for tool in VECTOR_SEARCH_TOOLS:
    resources.extend(tool.resources)
for tool in uc_toolkit.tools:
    # TODO: If the UC function includes dependencies like external connection or vector search, please include them manually.
    # See the TODO in the markdown above for more information.
    udf_name = tool.get("function", {}).get("name", "").replace("__", ".")
    resources.append(DatabricksFunction(function_name=udf_name))

input_example = {
    "input": [
        {
            "role": "user",
            "content": "Find the 1 TB SSD, check its stock in Chennai, and tell me the supplier’s city, country, and lead time."
        }
    ],
    "custom_inputs": {
        "session_id": "test-session"
    }
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        input_example=input_example,
        pip_requirements=[
            "databricks-openai",
            "backoff",
            f"databricks-connect=={get_distribution('databricks-connect').version}",
        ],
        resources=resources,
    )

## Evaluate the agent with [Agent Evaluation](https://docs.databricks.com/mlflow3/genai/eval-monitor)

You can edit the requests or expected responses in your evaluation dataset and run evaluation as you iterate your agent, leveraging mlflow to track the computed quality metrics.

Evaluate your agent with one of our [predefined LLM scorers](https://docs.databricks.com/mlflow3/genai/eval-monitor/predefined-judge-scorers), or try adding [custom metrics](https://docs.databricks.com/mlflow3/genai/eval-monitor/custom-scorers).

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, Safety, RetrievalRelevance, RetrievalGroundedness

eval_dataset = [
    {
        "inputs": {
            "input": [
                {
                    "role": "user",
                    "content": "Find the 1 TB SSD, check its stock in Chennai, and tell me the supplier’s city, country, and lead time."
                }
            ]
        },
        "expected_response": None
    },
    {
        "inputs": {
            "input": [
                {
                    "role": "user",
                    "content": "Find the 1 TB SSD, check its stock in Chennai, and tell me the supplier’s city, country, and lead time."
                },
                {
                    "role": "assistant",
                    "content": "To find the 1 TB SSD, we first need to search for it in the item master records using natural language description\n\n<uc_function_call>{\"id\":\"E11F49vgl94bYXfatuAI\",\"name\":\"main.scm_e2e_rag_lab.search_item_master\",\"arguments\":\"{\\\"search_query\\\":\\\"1 TB SSD\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"E11F49vgl94bYXfatuAI\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"item_document\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"Item ID: SSD001\\\\nItem name: 1 TB NVMe SSD\\\\nCategory: Storage\\\\nDescription: 1 TB M.2 NVMe solid state drive for computers and laptops\\\"],[\\\"RAM002\\\",\\\"Item ID: RAM002\\\\nItem name: 32 GB Server RAM\\\\nCategory: Memory\\\\nDescription: 32 GB DDR5 ECC memory module for servers\\\"],[\\\"RAM001\\\",\\\"Item ID: RAM001\\\\nItem name: 16 GB Laptop RAM\\\\nCategory: Memory\\\\nDescription: 16 GB DDR4 memory module for laptops\\\"]]}\"}</uc_function_result>\n\nTo check the stock of 1 TB SSD in Chennai, we first need to find the item ID of 1 TB SSD and then use the get_inventory function to find the stock in Chennai\n\n<uc_function_call>{\"id\":\"9w0p3WqCbXwWIZHoVVwm\",\"name\":\"main.scm_e2e_rag_lab.get_inventory\",\"arguments\":\"{\\\"p_item_id\\\":\\\"SSD001\\\",\\\"p_warehouse\\\":\\\"Chennai\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"9w0p3WqCbXwWIZHoVVwm\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"warehouse\\\",\\\"available_quantity\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"Chennai\\\",50]]}\"}</uc_function_result>\n\nTo find the supplier's city, country, and lead time for the 1 TB SSD item\n\n<uc_function_call>{\"id\":\"Q7tgfI8c2g8TvAkHErEG\",\"name\":\"main.scm_e2e_rag_lab.get_supplier\",\"arguments\":\"{\\\"p_item_id\\\":\\\"SSD001\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"Q7tgfI8c2g8TvAkHErEG\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"supplier_name\\\",\\\"supplier_city\\\",\\\"supplier_country\\\",\\\"lead_time_days\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"ABC Components\\\",\\\"Chennai\\\",\\\"India\\\",7]]}\"}</uc_function_result>\n\nThe 1 TB SSD (item ID: SSD001) is in stock in Chennai, with 50 units available. The supplier of this item is ABC Components, located in Chennai, India, and the lead time for delivery is 7 days."
                },
                {
                    "role": "user",
                    "content": "Find the 1 TB SSD, check its stock in Chennai, and tell me the supplier’s city, country, and lead time."
                }
            ]
        },
        "expected_response": None
    },
    {
        "inputs": {
            "input": [
                {
                    "role": "user",
                    "content": "Find the 1 TB SSD, check its stock in Chennai, and tell me the supplier’s city, country, and lead time."
                },
                {
                    "role": "assistant",
                    "content": "To find the 1 TB SSD, we first need to search for it in the item master records using natural language description\n\n<uc_function_call>{\"id\":\"E11F49vgl94bYXfatuAI\",\"name\":\"main.scm_e2e_rag_lab.search_item_master\",\"arguments\":\"{\\\"search_query\\\":\\\"1 TB SSD\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"E11F49vgl94bYXfatuAI\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"item_document\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"Item ID: SSD001\\\\nItem name: 1 TB NVMe SSD\\\\nCategory: Storage\\\\nDescription: 1 TB M.2 NVMe solid state drive for computers and laptops\\\"],[\\\"RAM002\\\",\\\"Item ID: RAM002\\\\nItem name: 32 GB Server RAM\\\\nCategory: Memory\\\\nDescription: 32 GB DDR5 ECC memory module for servers\\\"],[\\\"RAM001\\\",\\\"Item ID: RAM001\\\\nItem name: 16 GB Laptop RAM\\\\nCategory: Memory\\\\nDescription: 16 GB DDR4 memory module for laptops\\\"]]}\"}</uc_function_result>\n\nTo check the stock of 1 TB SSD in Chennai, we first need to find the item ID of 1 TB SSD and then use the get_inventory function to find the stock in Chennai\n\n<uc_function_call>{\"id\":\"9w0p3WqCbXwWIZHoVVwm\",\"name\":\"main.scm_e2e_rag_lab.get_inventory\",\"arguments\":\"{\\\"p_item_id\\\":\\\"SSD001\\\",\\\"p_warehouse\\\":\\\"Chennai\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"9w0p3WqCbXwWIZHoVVwm\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"warehouse\\\",\\\"available_quantity\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"Chennai\\\",50]]}\"}</uc_function_result>\n\nTo find the supplier's city, country, and lead time for the 1 TB SSD item\n\n<uc_function_call>{\"id\":\"Q7tgfI8c2g8TvAkHErEG\",\"name\":\"main.scm_e2e_rag_lab.get_supplier\",\"arguments\":\"{\\\"p_item_id\\\":\\\"SSD001\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"Q7tgfI8c2g8TvAkHErEG\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"supplier_name\\\",\\\"supplier_city\\\",\\\"supplier_country\\\",\\\"lead_time_days\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"ABC Components\\\",\\\"Chennai\\\",\\\"India\\\",7]]}\"}</uc_function_result>\n\nThe 1 TB SSD (item ID: SSD001) is in stock in Chennai, with 50 units available. The supplier of this item is ABC Components, located in Chennai, India, and the lead time for delivery is 7 days."
                },
                {
                    "role": "user",
                    "content": "Find the 1 TB SSD, check its stock in Chennai, and tell me the supplier’s city, country, and lead time."
                },
                {
                    "role": "assistant",
                    "content": "To find the 1 TB SSD, we first need to search for the item in the item master records.\n\n<uc_function_call>{\"id\":\"AnNOJb5F6mPjRuPaIH1j\",\"name\":\"main.scm_e2e_rag_lab.search_item_master\",\"arguments\":\"{\\\"search_query\\\":\\\"1 TB SSD\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"AnNOJb5F6mPjRuPaIH1j\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"item_document\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"Item ID: SSD001\\\\nItem name: 1 TB NVMe SSD\\\\nCategory: Storage\\\\nDescription: 1 TB M.2 NVMe solid state drive for computers and laptops\\\"],[\\\"RAM002\\\",\\\"Item ID: RAM002\\\\nItem name: 32 GB Server RAM\\\\nCategory: Memory\\\\nDescription: 32 GB DDR5 ECC memory module for servers\\\"],[\\\"RAM001\\\",\\\"Item ID: RAM001\\\\nItem name: 16 GB Laptop RAM\\\\nCategory: Memory\\\\nDescription: 16 GB DDR4 memory module for laptops\\\"]]}\"}</uc_function_result>\n\nTo check the stock of the 1 TB SSD in Chennai, we need to get the inventory for the item ID SSD001 in the Chennai warehouse.\n\n<uc_function_call>{\"id\":\"8KFBBO0hwe6LCEpSpyGi\",\"name\":\"main.scm_e2e_rag_lab.get_inventory\",\"arguments\":\"{\\\"p_item_id\\\":\\\"SSD001\\\",\\\"p_warehouse\\\":\\\"Chennai\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"8KFBBO0hwe6LCEpSpyGi\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"warehouse\\\",\\\"available_quantity\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"Chennai\\\",50]]}\"}</uc_function_result>\n\nI need to find the supplier of the 1 TB SSD to provide the supplier's city, country, and lead time.\n\n<uc_function_call>{\"id\":\"JsCuV8WsVWV333qSgqpo\",\"name\":\"main.scm_e2e_rag_lab.get_supplier\",\"arguments\":\"{\\\"p_item_id\\\":\\\"SSD001\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"JsCuV8WsVWV333qSgqpo\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"supplier_name\\\",\\\"supplier_city\\\",\\\"supplier_country\\\",\\\"lead_time_days\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"ABC Components\\\",\\\"Chennai\\\",\\\"India\\\",7]]}\"}</uc_function_result>\n\nThe 1 TB SSD (item ID: SSD001) is in stock in Chennai, with 50 units available. The supplier of this item is ABC Components, located in Chennai, India, and the lead time for delivery is 7 days."
                },
                {
                    "role": "user",
                    "content": "Find 16GB laptop ram, check the total stocks in all location and tell the supplier details"
                }
            ]
        },
        "expected_response": None
    },
    {
        "inputs": {
            "input": [
                {
                    "role": "user",
                    "content": "Find the 1 TB SSD, check its stock in Chennai, and tell me the supplier’s city, country, and lead time."
                },
                {
                    "role": "assistant",
                    "content": "To find the 1 TB SSD, we first need to search for it in the item master records using natural language description\n\n<uc_function_call>{\"id\":\"E11F49vgl94bYXfatuAI\",\"name\":\"main.scm_e2e_rag_lab.search_item_master\",\"arguments\":\"{\\\"search_query\\\":\\\"1 TB SSD\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"E11F49vgl94bYXfatuAI\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"item_document\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"Item ID: SSD001\\\\nItem name: 1 TB NVMe SSD\\\\nCategory: Storage\\\\nDescription: 1 TB M.2 NVMe solid state drive for computers and laptops\\\"],[\\\"RAM002\\\",\\\"Item ID: RAM002\\\\nItem name: 32 GB Server RAM\\\\nCategory: Memory\\\\nDescription: 32 GB DDR5 ECC memory module for servers\\\"],[\\\"RAM001\\\",\\\"Item ID: RAM001\\\\nItem name: 16 GB Laptop RAM\\\\nCategory: Memory\\\\nDescription: 16 GB DDR4 memory module for laptops\\\"]]}\"}</uc_function_result>\n\nTo check the stock of 1 TB SSD in Chennai, we first need to find the item ID of 1 TB SSD and then use the get_inventory function to find the stock in Chennai\n\n<uc_function_call>{\"id\":\"9w0p3WqCbXwWIZHoVVwm\",\"name\":\"main.scm_e2e_rag_lab.get_inventory\",\"arguments\":\"{\\\"p_item_id\\\":\\\"SSD001\\\",\\\"p_warehouse\\\":\\\"Chennai\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"9w0p3WqCbXwWIZHoVVwm\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"warehouse\\\",\\\"available_quantity\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"Chennai\\\",50]]}\"}</uc_function_result>\n\nTo find the supplier's city, country, and lead time for the 1 TB SSD item\n\n<uc_function_call>{\"id\":\"Q7tgfI8c2g8TvAkHErEG\",\"name\":\"main.scm_e2e_rag_lab.get_supplier\",\"arguments\":\"{\\\"p_item_id\\\":\\\"SSD001\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"Q7tgfI8c2g8TvAkHErEG\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"supplier_name\\\",\\\"supplier_city\\\",\\\"supplier_country\\\",\\\"lead_time_days\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"ABC Components\\\",\\\"Chennai\\\",\\\"India\\\",7]]}\"}</uc_function_result>\n\nThe 1 TB SSD (item ID: SSD001) is in stock in Chennai, with 50 units available. The supplier of this item is ABC Components, located in Chennai, India, and the lead time for delivery is 7 days."
                },
                {
                    "role": "user",
                    "content": "Find the 1 TB SSD, check its stock in Chennai, and tell me the supplier’s city, country, and lead time."
                },
                {
                    "role": "assistant",
                    "content": "To find the 1 TB SSD, we first need to search for the item in the item master records.\n\n<uc_function_call>{\"id\":\"AnNOJb5F6mPjRuPaIH1j\",\"name\":\"main.scm_e2e_rag_lab.search_item_master\",\"arguments\":\"{\\\"search_query\\\":\\\"1 TB SSD\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"AnNOJb5F6mPjRuPaIH1j\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"item_document\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"Item ID: SSD001\\\\nItem name: 1 TB NVMe SSD\\\\nCategory: Storage\\\\nDescription: 1 TB M.2 NVMe solid state drive for computers and laptops\\\"],[\\\"RAM002\\\",\\\"Item ID: RAM002\\\\nItem name: 32 GB Server RAM\\\\nCategory: Memory\\\\nDescription: 32 GB DDR5 ECC memory module for servers\\\"],[\\\"RAM001\\\",\\\"Item ID: RAM001\\\\nItem name: 16 GB Laptop RAM\\\\nCategory: Memory\\\\nDescription: 16 GB DDR4 memory module for laptops\\\"]]}\"}</uc_function_result>\n\nTo check the stock of the 1 TB SSD in Chennai, we need to get the inventory for the item ID SSD001 in the Chennai warehouse.\n\n<uc_function_call>{\"id\":\"8KFBBO0hwe6LCEpSpyGi\",\"name\":\"main.scm_e2e_rag_lab.get_inventory\",\"arguments\":\"{\\\"p_item_id\\\":\\\"SSD001\\\",\\\"p_warehouse\\\":\\\"Chennai\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"8KFBBO0hwe6LCEpSpyGi\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"warehouse\\\",\\\"available_quantity\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"Chennai\\\",50]]}\"}</uc_function_result>\n\nI need to find the supplier of the 1 TB SSD to provide the supplier's city, country, and lead time.\n\n<uc_function_call>{\"id\":\"JsCuV8WsVWV333qSgqpo\",\"name\":\"main.scm_e2e_rag_lab.get_supplier\",\"arguments\":\"{\\\"p_item_id\\\":\\\"SSD001\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"JsCuV8WsVWV333qSgqpo\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"supplier_name\\\",\\\"supplier_city\\\",\\\"supplier_country\\\",\\\"lead_time_days\\\"],\\\"rows\\\":[[\\\"SSD001\\\",\\\"ABC Components\\\",\\\"Chennai\\\",\\\"India\\\",7]]}\"}</uc_function_result>\n\nThe 1 TB SSD (item ID: SSD001) is in stock in Chennai, with 50 units available. The supplier of this item is ABC Components, located in Chennai, India, and the lead time for delivery is 7 days."
                },
                {
                    "role": "user",
                    "content": "Find 16GB laptop ram, check the total stocks in all location and tell the supplier details"
                },
                {
                    "role": "assistant",
                    "content": "To find the 16GB laptop RAM, we need to search the item master records using a natural-language description of the item.\n\n<uc_function_call>{\"id\":\"PlkbVvwDZei46hmvFSrd\",\"name\":\"main.scm_e2e_rag_lab.search_item_master\",\"arguments\":\"{\\\"search_query\\\":\\\"16GB laptop RAM\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"PlkbVvwDZei46hmvFSrd\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"item_document\\\"],\\\"rows\\\":[[\\\"RAM001\\\",\\\"Item ID: RAM001\\\\nItem name: 16 GB Laptop RAM\\\\nCategory: Memory\\\\nDescription: 16 GB DDR4 memory module for laptops\\\"],[\\\"RAM002\\\",\\\"Item ID: RAM002\\\\nItem name: 32 GB Server RAM\\\\nCategory: Memory\\\\nDescription: 32 GB DDR5 ECC memory module for servers\\\"],[\\\"SSD001\\\",\\\"Item ID: SSD001\\\\nItem name: 1 TB NVMe SSD\\\\nCategory: Storage\\\\nDescription: 1 TB M.2 NVMe solid state drive for computers and laptops\\\"]]}\"}</uc_function_result>\n\nTo get the supplier details for the 16GB laptop RAM\n\n<uc_function_call>{\"id\":\"QWGtiorXaMMY4gZErgBg\",\"name\":\"main.scm_e2e_rag_lab.get_supplier\",\"arguments\":\"{\\\"p_item_id\\\":\\\"RAM001\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"QWGtiorXaMMY4gZErgBg\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"item_id\\\",\\\"supplier_name\\\",\\\"supplier_city\\\",\\\"supplier_country\\\",\\\"lead_time_days\\\"],\\\"rows\\\":[[\\\"RAM001\\\",\\\"ABC Components\\\",\\\"Chennai\\\",\\\"India\\\",5]]}\"}</uc_function_result>\n\nThe 16 GB Laptop RAM (item ID: RAM001) is supplied by ABC Components, located in Chennai, India, with a lead time of 5 days. However, the total stock in all locations is not available from the previous output. \n\nTo get the total stock, I would need to know the warehouses where the item is stored. If you provide the list of warehouses, I can help you with the total stock. Alternatively, if you would like to check the stock in a specific warehouse, please let me know the warehouse name."
                },
                {
                    "role": "user",
                    "content": "Find 16GB laptop ram, check the available quantity and tell the supplier details"
                }
            ]
        },
        "expected_response": None
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input, "custom_inputs": {"session_id": "evaluation-session"}}),
    scorers=[RelevanceToQuery(), Safety()], # add more scorers here if they're applicable
)

# Review the evaluation results in the MLfLow UI (see console output)

## Perform pre-deployment validation of the agent
Before registering and deploying the agent, we perform pre-deployment checks via the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See [documentation](https://docs.databricks.com/machine-learning/model-serving/model-serving-debug.html#validate-inputs) for details

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "Find the 1 TB SSD, check its stock in Chennai, and tell me the supplier’s city, country, and lead time."}], "custom_inputs": {"session_id": "validation-session"}},
    env_manager="uv",
)

## Register the model to Unity Catalog

Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.

In [0]:
mlflow.set_registry_uri("databricks-uc")

# Confirm the catalog/schema exist and that you have CREATE MODEL permissions.
catalog = "main"
schema = "scm_e2e_rag_lab"
model_name = "scm_item_master_agent"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
)

## Deploy the agent

In [0]:
from databricks import agents
# NOTE: pass scale_to_zero=True to agents.deploy() to enable scale-to-zero for cost savings.
# This is not recommended for production workloads, as capacity is not guaranteed when scaled to zero.
# Scaled to zero endpoints may take extra time to respond when queried, while they scale back up.
agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, tags = {"endpointSource": "playground"},scale_to_zero=True)

## Next steps

After your agent is deployed, you can chat with it in AI playground to perform additional checks, share it with SMEs in your organization for feedback, or embed it in a production application. See [docs](https://docs.databricks.com/generative-ai/deploy-agent.html) for details